# Kompetenscirkeln: hämta data för hand

Normalt hämtas datan varje vecka av GitHub. Den här anteckningsboken är till för att göra samma hämtning manuellt, till exempel för ett annat yrkesområde.

**Så gör du:** välj **Körning → Kör alla** (Runtime → Run all). Det tar några minuter. Du får `dataset.json` (färdig datafil, läggs i `data/`) och `radata.json.gz` (rådata med annonstexter, läggs i `data/raw/`, ska inte delas vidare).

In [ ]:
import os
os.environ["FIELD_ID"] = "apaJ_2ja_LuF"  # Data/IT. Byt id för annat yrkesområde.

In [ ]:
%%writefile kompetens.py
"""Kompetenscirkeln: från rådata till färdig datafil. Bara standardbiblioteket.

Här finns hela matchningen (tidigare match.py + recompute.py):
- n-gram-matchning (samma ord i samma ordning), förkortningar skiftlägeskänsliga
- stoppord för etiketter som är vanliga svenska/engelska ord (Vi, Access, Management ...)
- kompetenser med samma visningsnamn slås ihop (fem varianter av Cybersäkerhet -> en)
- alternativa etiketter som 'Java, ramverk' matchas som hela fraser, inte som 'Java'
- varje annons matchas mot alla kompetenser, inte bara dem taxonomin kopplar till yrkesgruppen
- egna tillägg och synonymer ur tillagg.json (Kubernetes, Terraform ... som saknas i taxonomin)

build_dataset(raw) tar rådatan (det hamta.py hämtar) och ger det sidan läser.
"""
import collections, json, os, re

TOK = re.compile(r"[A-Za-zÅÄÖåäöÉéÜüØøÆæ0-9#+.]+")

STOP = {k.lower() for k in ["Vi", "Access", "Progress", "Client", "Server", "Logic", "MOST", "Ada", "Management", "Nätverk",
        "Arbetsmiljö", "E-post", "Friskvård", "Research", "Source", "Hälso- och sjukvård", "Ledningssystem", "Office",
        "Go", "Basic", "Swift", "Design", "Sales", "Support", "Service", "Teams"]}
RENAME = {"CI": "CI/CD", "Node": "Node.js", "Amazon Web Services": "AWS", "Google Cloud Platform": "Google Cloud"}

TOP_SKILLS, MIN_SHARE, MIN_OCC_ADS = 40, 0.02, 15
TILLAGG = os.path.join(os.path.dirname(os.path.abspath(__file__)), "tillagg.json")

METHOD = ("Kompetensvikt = andel aktuella annonser i Platsbanken för yrket som nämner kompetensen. Matchning på hela ord "
          "och fraser ur Arbetsförmedlingens kompetenslista; vanliga ord som 'vi' och 'access' är bortfiltrerade och "
          "dubbletter sammanslagna.")


def tokens(text):
    out = []
    for t in TOK.findall(text):
        t = t.strip(".")
        if t: out.append(t)
    return out


def grams(text, n=4):
    tk = tokens(text); orig = set(); low = set()
    for i in range(len(tk)):
        for k in range(1, n + 1):
            if i + k > len(tk): break
            g = " ".join(tk[i:i + k]); orig.add(g); low.add(g.lower())
    return orig, low


def term_key(t, force_cs=False):
    """(nyckel, skiftlägeskänslig). Korta termer och versalförkortningar matchas skiftlägeskänsligt."""
    tk = tokens(t)
    if not tk or len(tk) > 4: return None
    key = " ".join(tk)
    alnum = re.sub(r"\W", "", t)
    cs = force_cs or len(alnum) <= 3 or (alnum.isupper() and len(alnum) <= 6)
    return (key if cs else key.lower(), cs)


def display(label):
    head = label.split(",")[0].strip()
    head = re.sub(r"\s*\(.*?\)", "", head).strip()
    hs = [x for x in head.split("/") if x]
    if len(hs) > 1 and all(re.sub(r"\W", "", x).isupper() and len(x) <= 6 for x in hs):
        return head
    segs = [s.strip() for s in label.split("/")]
    for s in segs:
        a = re.sub(r"\W", "", s)
        if s and "," not in s and a.isupper() and 2 <= len(a) <= 6 and len(segs) > 1:
            return s
    return head.split("/")[0].strip() or label


def terms(s):
    lab = s["label"]
    own = re.sub(r"\(.*?\)", "", lab.split(",")[0]).strip()
    cands = [own] if "/" not in own else []
    for a in s.get("alt", []) + s.get("hidden", []):
        cands.append(a)
    out = []
    for c in cands:
        c = re.sub(r"\(.*?\)", "", c).strip()
        if "," in c:
            first = c.split(",")[0].strip()
            c = first if first.lower() == own.lower() else c.replace(",", " ")
        if len(c) < 2 or c.lower() in STOP: continue
        k = term_key(c)
        if k and k not in out: out.append(k)
    for c in s.get("extra", []):  # egna termer ur tillagg.json: redan granskade, ingen stopplista
        k = term_key(c, c in s.get("cs", []))
        if k and k not in out: out.append(k)
    return out


def load_extras(path=TILLAGG):
    if not path or not os.path.exists(path):
        return {"tillagg": [], "synonymer": {}}
    with open(path, encoding="utf-8") as f:
        x = json.load(f)
    return {"tillagg": x.get("tillagg", []), "synonymer": x.get("synonymer", {})}


def barometer_from_rows(rows):
    """Fältmappningen mot Yrkesbarometern. Ändras strukturen (dec 2026) är det här som ska justeras."""
    barometer, omgang = {}, None
    for r in rows:
        omgang = omgang or r.get("omgang")
        barometer.setdefault(str(r.get("ssyk")), {})[r.get("lan")] = {
            "yrke": r.get("yb_yrke"), "jobbmojligheter": r.get("jobbmojligheter"),
            "rekrytering": r.get("rekryteringssituation"), "prognos": r.get("prognos"),
            "text": r.get("text_jobbmojligheter")}
    return barometer, omgang


def build_dataset(raw, extras=None):
    """Rådata -> dataset. Deterministisk: samma rådata ger samma fil."""
    extras = load_extras() if extras is None else extras
    skills = {}
    for gid, lst in raw["skillsByGroup"].items():
        for s in lst: skills.setdefault(s["id"], dict(s))
    rename = {}
    for sid, syn in extras["synonymer"].items():
        if sid not in skills: continue  # taxonomin kan ha bytt id; då faller synonymen bort utan fel
        skills[sid]["extra"] = syn.get("termer", [])
        skills[sid]["cs"] = syn.get("cs", [])
        if syn.get("visa"): rename[sid] = syn["visa"]
    for t in extras["tillagg"]:
        skills["x_" + re.sub(r"\W+", "-", t["label"].lower())] = {
            "label": t["label"], "headline": t.get("headline"), "extra": [t["label"]] + t.get("termer", []),
            "cs": t.get("cs", []), "own": True}
    # ett id per visningsnamn
    canon, disp = {}, {}
    for sid in sorted(skills):
        s = skills[sid]
        d = rename.get(sid) or (s["label"] if s.get("own") else display(s["label"]))
        d = RENAME.get(d, d)
        canon[sid] = disp.setdefault(d.lower(), (sid, d, s.get("headline") or "Övrigt"))[0]
    terms_by = {sid: (terms(s) if not s.get("own") else [k for k in (term_key(c, c in s["cs"]) for c in s["extra"]) if k])
                for sid, s in skills.items()}
    all_ids = sorted(skills)

    def ad_hits(a, g):
        o, l = g
        hit = set(canon.get(x, x) for x in a["skillIds"] if x in skills)
        for sid in all_ids:
            if canon[sid] in hit: continue
            for key, cs in terms_by[sid]:
                if key in (o if cs else l): hit.add(canon[sid]); break
        return hit

    def shares(ads, hits):
        c = collections.Counter()
        for a in ads: c.update(hits[a["id"]])
        out = [{"skillId": k, "share": round(v / max(len(ads), 1), 4)} for k, v in c.items()]
        out = [x for x in out if x["share"] >= MIN_SHARE]
        return sorted(out, key=lambda x: (-x["share"], x["skillId"]))[:TOP_SKILLS]  # stabil ordning vid lika andel

    ads_by_group = collections.defaultdict(list)
    for a in raw["ads"]: ads_by_group[a["groupId"]].append(a)
    group_skills, occ_skills = {}, {}
    for g in raw["groups"]:
        ads = ads_by_group[g["id"]]
        hits = {a["id"]: ad_hits(a, grams(a["text"])) for a in ads}
        group_skills[g["id"]] = shares(ads, hits)
        by_occ = collections.defaultdict(list)
        for a in ads:
            if a.get("occupationId"): by_occ[a["occupationId"]].append(a)
        for o in raw["occupations"]:
            if o["groupId"] == g["id"] and len(by_occ[o["id"]]) >= MIN_OCC_ADS:
                occ_skills[o["id"]] = shares(by_occ[o["id"]], hits)

    barometer, omgang = barometer_from_rows(raw.get("barometerRows", []))
    return {
        "meta": {"demo": False, "generated": raw.get("fetched"), "field": raw.get("field"), "barometerOmgang": omgang,
                 "adsAnalyzed": len(raw["ads"]),
                 "adsInField": sum(len(ads_by_group[g["id"]]) for g in raw["groups"] if not g.get("external")),
                 "method": METHOD, "sources": ["JobTech Taxonomy", "JobSearch (Platsbanken)", "Yrkesbarometern"]},
        "groups": raw["groups"], "occupations": raw["occupations"],
        "skills": [{"id": cid, "label": d, "headline": h} for (cid, d, h) in disp.values()],  # alla, så att sökningen hittar dem
        "groupSkills": group_skills, "occSkills": occ_skills, "barometer": barometer,
    }


def write_dataset(ds, path):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(ds, f, ensure_ascii=False)


In [ ]:
%%writefile tillagg.json
{
  "_om": "Egna tillägg till Arbetsförmedlingens kompetenslista. 'tillagg' är kompetenser som saknas helt i taxonomin. 'synonymer' ger extra söktermer (och ev. nytt visningsnamn, 'visa') till en kompetens som finns, nyckeln är taxonomins id. Termer i 'cs' matchas skiftlägeskänsligt. Listan bör granskas av vägledare och stämmas av med JobTech.",
  "tillagg": [
    {"label": "Kubernetes", "headline": "Applikationsplattformar", "termer": ["K8s", "k8s"]},
    {"label": "OpenShift", "headline": "Applikationsplattformar"},
    {"label": "Terraform", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "Ansible", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "GitHub", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "GitLab", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "Grafana", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "Prometheus", "headline": "Programmerings- och systemutvecklingsverktyg", "cs": ["Prometheus"]},
    {"label": "DevOps", "headline": "Styr- och utvecklingsmodeller"},
    {"label": "SAFe", "headline": "Styr- och utvecklingsmodeller", "cs": ["SAFe"]},
    {"label": "MLOps", "headline": "Styr- och utvecklingsmodeller"},
    {"label": "React", "headline": "Ramverk", "termer": ["React.js", "ReactJS"], "cs": ["React"]},
    {"label": "Next.js", "headline": "Ramverk", "termer": ["NextJS"]},
    {"label": "GraphQL", "headline": "Datorspråk"},
    {"label": "Power BI", "headline": "Programmerings- och systemutvecklingsverktyg", "termer": ["PowerBI"]},
    {"label": "Databricks", "headline": "Applikationsplattformar"},
    {"label": "Snowflake", "headline": "Databaser", "cs": ["Snowflake"]},
    {"label": "dbt", "headline": "Programmerings- och systemutvecklingsverktyg"},
    {"label": "Airflow", "headline": "Programmerings- och systemutvecklingsverktyg", "cs": ["Airflow"]},
    {"label": "Apache Spark", "headline": "Applikationsplattformar", "termer": ["Spark", "PySpark"], "cs": ["Spark"]},
    {"label": "PyTorch", "headline": "Ramverk"},
    {"label": "TensorFlow", "headline": "Ramverk"},
    {"label": "Språkmodeller", "headline": "Övriga kompetenser", "termer": ["LLM", "LLMs", "LLM:er", "språkmodeller", "large language models"]},
    {"label": "Figma", "headline": "Programvara för digitalt skapande"},
    {"label": "Entra ID", "headline": "Programmerings- och systemutvecklingsverktyg", "termer": ["Microsoft Entra", "Azure AD", "Azure Active Directory"]},
    {"label": "Microsoft 365", "headline": "Applikationsplattformar", "termer": ["M365", "Office 365", "O365"]},
    {"label": "ServiceNow", "headline": "Applikationsplattformar"}
  ],
  "synonymer": {
    "HKgZ_SWa_vn8": {"visa": "Azure", "termer": ["Azure", "Microsoft Azure"]},
    "hYyY_NPi_H65": {"visa": "Kafka", "termer": ["Kafka"]},
    "c6uA_XkH_gpt": {"visa": "Intune", "termer": ["Intune"]}
  }
}


In [ ]:
%%writefile hamta.py
"""Kompetenscirkeln: hämtar Arbetsförmedlingens öppna data och skriver en färdig datafil. En körning, inga beroenden.

    python3 pipeline/hamta.py                         # -> data/dataset.json + data/raw/radata.json.gz
    python3 pipeline/hamta.py --utan-radata           # bara datafilen (så körs veckojobbet på GitHub)
    FIELD_ID=<id> python3 pipeline/hamta.py           # annat yrkesområde än Data/IT

Kräver nät mot jobtechdev.se (vanlig dator, Colab eller GitHub Actions) och Python 3.9+.
Skriptet avbryter utan att skriva något om hämtningen ser trasig ut, så att en dålig körning inte publiceras.
"""
import argparse, gzip, json, os, sys, time, urllib.error, urllib.parse, urllib.request
from datetime import datetime, timezone

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import kompetens

ROOT = os.path.join(os.path.dirname(os.path.abspath(__file__)), "..")
CFG = {
    "FIELD_ID": os.environ.get("FIELD_ID", "apaJ_2ja_LuF"),  # Data/IT
    "TAXONOMY": os.environ.get("TAXONOMY_BASE", "https://taxonomy.api.jobtechdev.se"),
    "JOBSEARCH": os.environ.get("JOBSEARCH_BASE", "https://jobsearch.api.jobtechdev.se"),
    "BAROMETER_URL": os.environ.get("BAROMETER_URL", "https://data.jobtechdev.se/prognoser/yrkesbarometer.json"),
    "MAX_ADS_PER_GROUP": 2000, "MAX_ADS_EXTERNAL_GROUP": 500, "INCLUDE_EXTERNAL": True, "MAX_EXTERNAL_GROUPS": 12,
    "DELAY": float(os.environ.get("DELAY", "0.15")), "RAW_TEXT_MAX": 6000,
    # rimlighetsgränser: under dessa skrivs ingen fil
    "MIN_GROUPS": 3, "MIN_ADS": int(os.environ.get("MIN_ADS", "500")),
}


def get_json(url, retries=4):
    for attempt in range(1, retries + 1):
        try:
            req = urllib.request.Request(url, headers={"accept": "application/json", "user-agent": "kompetenscirkeln"})
            with urllib.request.urlopen(req, timeout=60) as r:
                return json.loads(r.read().decode("utf-8"))
        except urllib.error.HTTPError as e:
            if e.code < 500 and e.code != 429 or attempt >= retries:
                raise RuntimeError(f"{url}\n  -> HTTP {e.code}")
        except Exception as e:
            if attempt >= retries:
                raise RuntimeError(f"{url}\n  -> {e}")
        time.sleep(2 * attempt)


def gql(query):
    data = get_json(f"{CFG['TAXONOMY']}/v1/taxonomy/graphql?query={urllib.parse.quote(query)}")
    if data.get("errors"):
        raise RuntimeError("GraphQL-fel: " + json.dumps(data["errors"])[:500])
    time.sleep(CFG["DELAY"])
    return data["data"]


def fetch_field(fid):
    d = gql('{ concepts(id: "%s") { id preferred_label narrower(type: "ssyk-level-4") { id preferred_label ssyk_code_2012 '
            'narrower(type: "occupation-name") { id preferred_label substitutes { id preferred_label substitutability_percentage '
            'broader(type: "ssyk-level-4") { id preferred_label ssyk_code_2012 } } } } } }' % fid)
    if not d.get("concepts"):
        raise RuntimeError(f"Hittade inget yrkesområde med id {fid}")
    return d["concepts"][0]


def fetch_group_skills(gid):
    d = gql('{ concepts(id: "%s") { related(type: "skill") { id preferred_label alternative_labels hidden_labels '
            'broader(type: "skill-headline") { preferred_label } } } }' % gid)
    return (d.get("concepts") or [{}])[0].get("related") or []


def fetch_ads(gid, max_ads):
    ads, seen, total, offset = [], set(), 0, 0
    while offset < max_ads:
        limit = min(100, max_ads - offset)
        j = get_json(f"{CFG['JOBSEARCH']}/search?occupation-group={gid}&limit={limit}&offset={offset}")
        total = (j.get("total") or {}).get("value", 0)
        hits = j.get("hits") or []
        for h in hits:
            if h.get("id") in seen:  # sidindelningen kan ge samma annons två gånger när nya publiceras under hämtningen
                continue
            seen.add(h.get("id"))
            sk = ((h.get("must_have") or {}).get("skills") or []) + ((h.get("nice_to_have") or {}).get("skills") or [])
            text = f"{h.get('headline') or ''}\n{(h.get('description') or {}).get('text') or ''}"
            ads.append({
                "id": h.get("id"), "published": h.get("publication_date"),
                "region": (h.get("workplace_address") or {}).get("region"),
                "occupationId": (h.get("occupation") or {}).get("concept_id"),
                "text": text[: CFG["RAW_TEXT_MAX"]],
                "skillIds": [s.get("concept_id") for s in sk if s.get("concept_id")],
                "groupId": gid,
            })
        time.sleep(CFG["DELAY"])
        if offset + 100 >= total or not hits:
            break
        offset += 100
    return total, ads


def fetch_raw():
    print("• Hämtar yrkesområde", CFG["FIELD_ID"], "från taxonomin …")
    field = fetch_field(CFG["FIELD_ID"])
    print(f"• {field['preferred_label']}: {len(field['narrower'])} yrkesgrupper")
    groups, occupations, internal, external = [], [], set(), {}
    for g in field["narrower"]:
        groups.append({"id": g["id"], "label": g["preferred_label"], "ssyk": g["ssyk_code_2012"], "external": False})
        for o in g.get("narrower") or []:
            internal.add(o["id"])
    for g in field["narrower"]:
        for o in g.get("narrower") or []:
            subs = o.get("substitutes") or []
            occupations.append({"id": o["id"], "label": o["preferred_label"], "groupId": g["id"],
                                "substitutes": [{"id": s["id"], "pct": s.get("substitutability_percentage")} for s in subs]})
            if not CFG["INCLUDE_EXTERNAL"]:
                continue
            for s in subs:
                sg = (s.get("broader") or [None])[0]
                if s["id"] in internal or not sg:
                    continue
                e = external.setdefault(sg["id"], {"group": sg, "occupations": {}, "refs": 0})
                e["refs"] += 1
                e["occupations"][s["id"]] = s["preferred_label"]
    chosen = sorted(external.values(), key=lambda e: -e["refs"])[: CFG["MAX_EXTERNAL_GROUPS"]]
    for e in chosen:
        g = e["group"]
        groups.append({"id": g["id"], "label": g["preferred_label"], "ssyk": g["ssyk_code_2012"], "external": True})
        for oid, label in e["occupations"].items():
            occupations.append({"id": oid, "label": label, "groupId": g["id"], "substitutes": [], "external": True})
    if chosen:
        print(f"• + {len(chosen)} närliggande yrkesgrupper utanför området (via utbytbarhet)")

    raw = {"field": {"id": field["id"], "label": field["preferred_label"]}, "fetched": datetime.now(timezone.utc).isoformat(),
           "groups": groups, "skillsByGroup": {}, "ads": [], "barometerRows": [], "occupations": occupations}
    for g in groups:
        skills = fetch_group_skills(g["id"])
        raw["skillsByGroup"][g["id"]] = [
            {"id": s["id"], "label": s["preferred_label"], "alt": s.get("alternative_labels") or [],
             "hidden": s.get("hidden_labels") or [],
             "headline": ((s.get("broader") or [{}])[0] or {}).get("preferred_label")} for s in skills]
        total, ads = fetch_ads(g["id"], CFG["MAX_ADS_EXTERNAL_GROUP"] if g["external"] else CFG["MAX_ADS_PER_GROUP"])
        raw["ads"].extend(ads)
        g["adCount"], g["adsAnalyzed"] = total, len(ads)
        per_occ = {}
        for ad in ads:
            if ad["occupationId"]:
                per_occ[ad["occupationId"]] = per_occ.get(ad["occupationId"], 0) + 1
        for o in occupations:
            if o["groupId"] == g["id"]:
                o["adCount"] = per_occ.get(o["id"], 0)
        print(f"• [{g['ssyk']}] {g['label']}: {len(skills)} kompetenser i taxonomin, {len(ads)}/{total} annonser")

    print("• Hämtar Yrkesbarometern …")
    ssyks = {str(g["ssyk"]) for g in groups}
    raw["barometerRows"] = [r for r in get_json(CFG["BAROMETER_URL"]) if str(r.get("ssyk")) in ssyks]
    return raw


def check(raw, ds):
    """Rimlighetskontroll. Returnerar en lista med fel; tom lista = godkänt."""
    fel = []
    if len(raw["groups"]) < CFG["MIN_GROUPS"]:
        fel.append(f"bara {len(raw['groups'])} yrkesgrupper")
    if len(raw["ads"]) < CFG["MIN_ADS"]:
        fel.append(f"bara {len(raw['ads'])} annonser (gräns {CFG['MIN_ADS']})")
    if not any(ds["groupSkills"].values()):
        fel.append("inga kompetenser matchade")
    if not ds["barometer"] or not ds["meta"]["barometerOmgang"]:
        fel.append("Yrkesbarometern är tom eller har bytt fältnamn")
    elif not any(v.get("jobbmojligheter") for lan in ds["barometer"].values() for v in lan.values()):
        fel.append("Yrkesbarometern saknar fältet jobbmojligheter")
    return fel


def main():
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--ut", default=os.path.join(ROOT, "data", "dataset.json"))
    ap.add_argument("--radata", default=os.path.join(ROOT, "data", "raw", "radata.json.gz"))
    ap.add_argument("--utan-radata", action="store_true", help="spara inte rådatan (annonstexterna)")
    a = ap.parse_args()
    raw = fetch_raw()
    ds = kompetens.build_dataset(raw)
    fel = check(raw, ds)
    if fel:
        sys.exit("✗ Hämtningen ser trasig ut, inget skrevs: " + "; ".join(fel))
    kompetens.write_dataset(ds, a.ut)
    if not a.utan_radata:
        os.makedirs(os.path.dirname(os.path.abspath(a.radata)), exist_ok=True)
        with gzip.open(a.radata, "wt", encoding="utf-8") as f:
            json.dump(raw, f, ensure_ascii=False)
    print(f"• Klart: {len(ds['occupations'])} yrken, {len(ds['skills'])} kompetenser, {len(raw['ads'])} annonser -> {a.ut}")


if __name__ == "__main__":
    main()


In [ ]:
!python hamta.py --ut dataset.json --radata radata.json.gz

In [ ]:
try:
    from google.colab import files
    files.download("dataset.json")
    files.download("radata.json.gz")
except ImportError:
    print("Inte i Colab: filerna ligger i samma mapp som anteckningsboken.")